# Homework: Batch effects and multiple testing on the Pasilla dataset

---

## Background

The pasilla dataset contains 7 samples: 4 untreated and 3 treated with RNAi against the pasilla splicing factor. Samples were sequenced using two different protocols (single-read and paired-end), which introduces a **batch effect**.

## Objective

Compare p-value histograms under four scenarios:

| Scenario | Batch correction | Multiple hypothesis correction |
|----------|:---:|:---:|
| A | - | - |
| B | - | + |
| C | + | - |
| D | + | + |

For each scenario, you will:
1. Run a differential expression analysis (treated vs. untreated)
2. Plot the p-value histogram
3. Count the number of significant genes

Then you will interpret the results answering 3 questions.

**Warning: Adjusted p-values don't have a meaningful histogram shape (e.g., BH correction multiplies raw p-values by m/i, which distorts the distribution and pushes most values toward 1). The p-value histogram diagnostic (Scenarios A–F) only applies to raw p-values.**

---

## Setup

In [ ]:
# !pip install inmoose statsmodels

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['font.size'] = 12

## Step 1: Load and prepare the data

Load the pasilla dataset, filter low-count genes, and compute log2-CPM.

In [ ]:
from inmoose.data.pasilla import pasilla

pas = pasilla()
counts = pas.X.T  # genes x samples
anno = pas.obs
gene_names = pas.var.index.tolist()

print(f"Samples: {counts.shape[1]}")
print(anno[['condition', 'type']])

# Filter low-count genes
keep = counts.sum(axis=1) > 10
counts_filt = counts[keep]
gene_names_filt = [g for g, k in zip(gene_names, keep) if k]
print(f"\nKept {keep.sum()} of {len(keep)} genes")

# Log2-CPM
lib_sizes = counts_filt.sum(axis=0)
log_cpm = np.log2(counts_filt / lib_sizes * 1e6 + 1)

## Step 2: Helper function for p-value histograms

In [ ]:
def plot_pvalue_histogram(pvals, title='', color='blue', ax=None):
    """Plot a p-value histogram"""
    if ax is None:
        fig, ax = plt.subplots()
    pvals_clean = pvals[~np.isnan(pvals)]
    ax.hist(pvals_clean, bins=50, range=(0, 1), color=color, edgecolor='white', alpha=0.8)
    ax.set_xlabel('P-value')
    ax.set_ylabel('Frequency')
    ax.set_title(title, fontweight='bold')
    ax.set_xlim(0, 1)
    ax.legend(fontsize=9)
    return ax

---

## Exercise 1: Run a per-gene t-test WITHOUT batch correction

For each gene, run a two-sample t-test comparing treated vs. untreated samples using log2-CPM values. Ignore the sequencing type (batch) for now.

**Task:** Fill in the code below to compute a p-value for each gene.

In [ ]:
# --- IMPLEMENTED ---
from scipy.stats import ttest_ind
# --- END ---


# Indices for treated and untreated samples
treated = anno['condition'].values == 'treated'
untreated = anno['condition'].values == 'untreated'

# YOUR CODE HERE: compute p-values for each gene
pvals_no_batch = np.zeros(log_cpm.shape[0]) # genex x probes
for i in range(log_cpm.shape[0]):
    # TODO: run a t-test between treated and untreated for gene i
    # --- IMPLEMENTED ---
    result = ttest_ind(log_cpm[i, treated], log_cpm[i, untreated])
    pvals_no_batch[i] = result.pvalue
    # --- END ---
    pass

In [ ]:
plot_pvalue_histogram(pvals_no_batch)

---

## Exercise 2: Run a per-gene test WITH batch correction

Use a linear model (OLS) with both `type` (batch) and `condition` in the design to account for the sequencing type.

**Task:** Fit `~ type + condition` per gene using `statsmodels.OLS`, extract the p-value for the `condition` coefficient.

In [ ]:
log_cpm.shape # same wartości genex x probes
anno # adnotacja próbek, wykorzystuje to do sotrotwania 
# dmatrix ? 
design # to nam mówi - jakie wartości przyjmują zmienne na daną próbę (próby x wartości na próbę)
# teraz każdy gen ekspresjoonowany to jest jeden eksperyemnt  
condition_col # tutaj wydobywmy indeks kolumny o danej warunku - u nas condition 


In [ ]:
import statsmodels.api as sm
from patsy import dmatrix

# comment: tutaj ustalamy relacje pomiędzy zmiennymi 
design = dmatrix("~ type + condition", data=anno) # (probes x 3 - type, condition, intercept ) 
condition_col = design.design_info.column_names.index('condition[T.untreated]')

# YOUR CODE HERE: compute p-values using OLS with batch correction
pvals_batch = np.zeros(log_cpm.shape[0])
for i in range(log_cpm.shape[0]):
    # TODO: fit OLS model and extract p-value for condition coefficient
    # --- IMPLEMENTED ---
    ols_model = sm.OLS(log_cpm[i, :].T, design) # shape (probes, )
    results = ols_model.fit()
    pvals_batch[i] = results.pvalues[condition_col]
    # --- END ---
    pass

In [ ]:
plot_pvalue_histogram(pvals_batch)

---

## Exercise 3: Apply multiple testing correction

Apply Benjamini-Hochberg FDR correction to both sets of p-values.

**Task:** Use `multipletests` from `statsmodels` to compute adjusted p-values for both `pvals_no_batch` and `pvals_batch`.

In [ ]:
# TODO - sprawdzić w jaki sposób to interpretować 
pvals_no_batch_adj[1] # który tutaj przechodzi 

In [ ]:
from statsmodels.stats.multitest import multipletests

# YOUR CODE HERE: apply BH correction to both sets of p-values
pvals_no_batch_adj = multipletests(pvals_no_batch, alpha=0.001)
pvals_batch_adj = multipletests(pvals_batch, alpha=0.001)

# TODO - here it is meaningless 
# TODO - check - we obtain only one 
plot_pvalue_histogram(pvals_no_batch_adj[1])
plot_pvalue_histogram(pvals_batch_adj[1])

---

## Exercise 4: Compare all four scenarios

**Task:** Plot p-value histograms for all four scenarios in a 2×2 grid and count the number of significant genes (P < 0.05) in each.

**Remember: Adjusted p-values don't have a meaningful histogram shape like in the discussed raw p-value scenarios!**

In [ ]:
# YOUR CODE HERE: create a 2x2 grid of p-value histograms
# Top row: no batch correction (raw p-values, adjusted p-values)
# Bottom row: with batch correction (raw p-values, adjusted p-values)

---

## Exercise 5: Interpretation

**Task:** Print a summary table and answer the questions below.

In [ ]:
# YOUR CODE HERE: print a summary table with the number of significant genes for each scenario

### Questions

**Q1:** Compare the p-value histograms of scenarios A and C (no MHT correction). How does including batch in the model change the shape of the histogram? What does this tell you about the batch effect in this dataset?

**Q2:** Compare scenarios A and B. What does multiple testing correction do to the p-value distribution? Why do most adjusted p-values pile up near 1?

**Q3:** Which scenario (A–D) gives you the most trustworthy set of significant genes? Why?

**A1:** TODO

**A2:** TODO

**A3:** TODO